# SLA, Capacity, and Commercials

> **The story:** In 1961, John Little published the proof behind $L = \lambda W$, turning a relationship observed in queues into a general law. The formula is simple enough to invite misuse: averages describe a stable system, while customer deadlines, retries, long requests, quota, and support commitments live in the tails. Riverside House needs the law, but it also needs the discipline to know what the law cannot promise.
>
> **Where you are:** Discovery supplied a frozen synthetic engagement with 620 expected sessions per business day, a 900-session high case, request-type latency targets, a 14,000 USD monthly target, an 18,000 USD planning ceiling, and weekday support hours. Architecture and identity work established that authorization and residency are constraints, not cost knobs. You now have to decide what can be offered, what must be measured, and what remains commercially unsafe to claim.
>
> **Notation:** $\lambda$ is arrival rate; $W$ is time in system; $L$ is in-flight concurrency; $H$ is fractional headroom; $T_{in}$ and $T_{out}$ are billed input and output tokens; $r$ is retry rate; $h$ is realized cache-hit rate; RPM and TPM are requests and tokens per minute; $C_m$ is monthly cost.

> **What you finished last time:** the Riverside FDE route established customer constraints, evidence labels, and a frozen case.
> **What this notebook delivers:** an auditable capacity and commercial planning envelope plus reusable SLA and decision templates.
> **Prerequisite for rollout:** live load, quota, price, regional, failover, and support evidence must replace the external-validation placeholders.

## 0 - The Challenge and Evidence Contract

> **The mission:** Riverside House - propose a service tier and planning envelope without pretending an average is capacity, an estimate is a quote, or a target is measured performance.

**What we know so far:**
- `[Modeled]` expected demand is 620 sessions per business day; the high case is 900.
- `[Modeled]` the operating target is 14,000 USD per month; 18,000 USD is the hard planning ceiling before exception review.
- `[Policy constraint]` forbidden access and duplicate workflow commits have a target of zero.
- **But we still cannot convert those statements into a defensible SLA, capacity reservation, or price.**

**What's blocking us:** daily averages hide bursts and tails; the frozen case has sessions and peak requests but no canonical requests-per-session; prices, quota, regional capacity, failover, and staffing are unvalidated. Any single-number answer would bury those gaps.

**What this chapter unlocks:** a scenario range, explicit contradictions, capacity and cost drivers, and a tier decision whose unsupported parts remain visibly unsupported.

```mermaid
flowchart LR
    A["Frozen case and traces"] --> B["Failure: average-only plan"]
    B --> C["Scenario ranges"]
    C --> D["Capacity and quota"]
    D --> E["Full cost attribution"]
    E --> F["SLA tier and commercial decision"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

| Topic | Coverage | Boundary |
|---|---|---|
| Arrival, concurrency, tokens, cache, retries, headroom | Built | Scenario model, not production measurement |
| Model, infrastructure, retrieval/storage, observability, software, support cost | Built | Illustrative rates require validation |
| SLA tiers and error budgets | Built | Proposed operating model, not legal language |
| Queueing simulation and production load test | Named only | Requires an agreed traffic distribution and executable environment |
| Live cloud price, quota, capacity, and failover | Named only | External validation required |

In [ ]:
# -- Load frozen and measured synthetic evidence ---------------------------
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (
        path
        for path in candidates
        if (path / "AUTHORING_GUIDE.md").is_file()
        and (path / "learning" / "fde" / "shared").is_dir()
    ),
    None,
)
assert REPO_ROOT is not None, "Run from inside the ai-portfolio repository."
CHAPTER_DIR = REPO_ROOT / "learning" / "fde" / "05-sla-capacity-and-commercials"
CASE_PATH = REPO_ROOT / "learning" / "fde" / "shared" / "fixtures" / "riverside-engagement-v1.json"
FACTS_PATH = REPO_ROOT / "learning" / "fde" / "shared" / "fixtures" / "expected-facts-v1.json"
TRACES_PATH = REPO_ROOT / "learning" / "ai-engineer" / "shared" / "latency-cost" / "request-traces.jsonl"
COST_INPUT_PATH = CHAPTER_DIR / "templates" / "cost-input-template.csv"

case = json.loads(CASE_PATH.read_text(encoding="utf-8"))
expected_facts = json.loads(FACTS_PATH.read_text(encoding="utf-8"))
traces = [json.loads(line) for line in TRACES_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
assumption_rows = case["demand_cost_and_sla_assumptions"]["assumptions"]
assumptions = {row["name"]: row for row in assumption_rows}
service_targets = case["demand_cost_and_sla_assumptions"]["service_targets"]
fact_by_id = {fact["fact_id"]: fact for fact in expected_facts["facts"]}

successful = [trace for trace in traces if trace["outcome"] == "success"]
trace_cost = sum(trace["total_cost_microusd"] for trace in traces) / 1_000_000
trace_attempts = sum(stage["stage_name"] == "generation" for trace in traces for stage in trace["stages"])
trace_generation_requests = sum(any(stage["stage_name"] == "generation" for stage in trace["stages"]) for trace in traces)

checks = {
    "fixture_version_matches": expected_facts["fixture_version"] == case["fixture_version"],
    "expected_sessions_preserved": assumptions["average_daily_sessions"]["value"] == fact_by_id["FACT-RIV-019"]["expected_value"],
    "budget_preserved": assumptions["monthly_service_budget_target"]["value"] == fact_by_id["FACT-RIV-021"]["expected_value"],
    "all_demand_inputs_modeled": all(row["evidence_class"] == "modeled_assumption" for row in assumption_rows),
    "trace_ids_unique": len({trace["request_id"] for trace in traces}) == len(traces),
}
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(checks.values()), "Stop: source evidence failed integrity checks."
print(f"[Modeled input] frozen fixture: {case['fixture_version']}")
print(f"[Measured: synthetic fixture] requests: {len(traces)}, successful: {len(successful)}, observed cost: ${trace_cost:.6f}")
print(f"[Measured: synthetic fixture] generation-attempt amplification: {trace_attempts / trace_generation_requests:.2f}x")
print("LIMIT: five synthetic traces teach attribution; they do not estimate production percentiles or price.")

## 1 - Make the Average-Only Plan Fail

A first pass divides 620 sessions by the ten covered hours and quietly assumes one request per session. That produces a clean number. Clean is not the same as usable.

**Predict:** which result invalidates the plan first?
1. The stated peak is above the one-request-per-session average.
2. Adding the chapter's expected requests-per-session makes implied average requests exceed the frozen peak.
3. Both - because the units and demand definitions are not reconciled.

```mermaid
flowchart LR
    A["620 sessions/day"] --> B["Divide by 10 hours"]
    B --> C["62 sessions/hour"]
    C --> D["Hidden assumption: 1 request/session"]
    D --> E["Frozen peak: 90 requests/hour"]
    E --> F["Reconcile units before sizing"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Divide daily sessions by hours and call it peak RPS | Burst shape disappears |
| Wrong | Mix sessions/hour and requests/hour without a conversion contract | The model can become internally impossible |
| Right | Reconcile definitions; until then use the larger rate and show the conflict | Undercapacity is not hidden by arithmetic |

**Quick Health Check:** fail if a stated peak is lower than an implied average, if units differ, or if the request/session conversion has no validation owner.

In [ ]:
# ── Expose and Plot the Average-Only Failure ──────────────────────────────────
covered_hours_per_day = 10
expected_sessions = assumptions["average_daily_sessions"]["value"]
stated_peak_rph = assumptions["peak_arrival_rate"]["value"]
chapter_requests_per_session = 3.4  # [Modeled] Exercise input; not a frozen customer fact.

naive_rph = expected_sessions / covered_hours_per_day
implied_rph = expected_sessions * chapter_requests_per_session / covered_hours_per_day
planning_rph = max(stated_peak_rph, implied_rph)
print(f"[Modeled] naive one-request/session average: {naive_rph:.1f} requests/hour")
print(f"[Modeled] frozen stated peak: {stated_peak_rph:.1f} requests/hour")
print(f"[Modeled] implied average at {chapter_requests_per_session:.1f} requests/session: {implied_rph:.1f} requests/hour")
print(f"[Modeled] conservative planning rate before reconciliation: {planning_rph:.1f} requests/hour")
if implied_rph > stated_peak_rph:
    prediction_result = "prediction 3 confirmed - the inputs use unreconciled demand definitions"
    next_action = "size from the larger rate temporarily and assign Operations to reconcile telemetry"
else:
    prediction_result = "prediction 1 confirmed - the frozen peak exceeds the modeled implied average"
    next_action = "retain the frozen peak provisionally and validate requests per session plus burst shape"
print(f"RESULT: {prediction_result}.")
print(f"ACTION: {next_action}.")

labels = ["Naive average", "Frozen peak", "Implied average", "Planning rate"]
values = [naive_rph, stated_peak_rph, implied_rph, planning_rph]
colors = ["#1d4ed8", "#b45309", "#b91c1c", "#15803d"]
fig, ax = plt.subplots(figsize=(9, 4.5), facecolor="#1a1a2e")
ax.set_facecolor("#1a1a2e")
bars = ax.bar(labels, values, color=colors)
ax.bar_label(bars, fmt="%.1f", color="white")
ax.set_ylabel("Requests per hour", color="white")
ax.set_title("[Modeled] One average cannot reconcile session and request demand", color="white")
ax.tick_params(colors="white", axis="x", rotation=15)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

average_plan_checks = {
    "positive_covered_hours": covered_hours_per_day > 0,
    "positive_requests_per_session": chapter_requests_per_session > 0,
    "planning_rate_not_below_any_input": planning_rph >= max(stated_peak_rph, implied_rph),
    "reconciliation_flag_matches_inputs": (implied_rph > stated_peak_rph) == ("unreconciled" in prediction_result),
}
for name, passed in average_plan_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert average_plan_checks["planning_rate_not_below_any_input"]
print("LIMIT: this conservative maximum is a temporary planning rule, not a measured arrival distribution.")

## 2 - Build Scenario Ranges, Not a Decimal-Heavy Forecast

Low, expected, and high are coherent planning stories, not confidence intervals. Each story combines frozen ranges with chapter-owned assumptions that stay visibly external-validation inputs.

```mermaid
flowchart TD
    A["Frozen ranges"] --> D["Scenario table"]
    B["Chapter assumptions"] --> D
    C["Request mix"] --> D
    D --> E["Monthly requests and tokens"]
    E --> F["Capacity and cost inputs"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Treat low/expected/high as p10/p50/p90 | No probability model was fitted |
| Wrong | Apply cache eligibility directly as hit rate | Eligibility does not prove safe, reusable hits |
| Right | Separate eligible rate from realization and authorization-scoped keying | Cache savings remain bounded by policy |

**Quick Health Check:** scenario ordering, request-mix sum, token positivity, cache bounds, retry bounds, and planning rate must all pass.

In [ ]:
# -- Build, plot, and check demand/token scenarios -------------------------
request_mix = assumptions["request_mix"]["value"]
business_days_per_month = 22
scenario_inputs = {
    "low": {"sessions_per_day": 450, "stated_peak_rph": 60, "requests_per_session": 2.5, "input_tokens": 900, "output_tokens": 90, "cache_eligible": 0.08, "cache_realization": 0.45, "retry_rate": 0.01, "support_hours": 48, "budget": 11000, "headroom": 0.25, "policy_s": 3.5, "continuation_s": 8.0, "workflow_s": 5.0},
    "expected": {"sessions_per_day": 620, "stated_peak_rph": 90, "requests_per_session": 3.4, "input_tokens": 1800, "output_tokens": 220, "cache_eligible": 0.18, "cache_realization": 0.65, "retry_rate": 0.025, "support_hours": 80, "budget": 14000, "headroom": 0.35, "policy_s": 5.0, "continuation_s": 14.0, "workflow_s": 8.0},
    "high": {"sessions_per_day": 900, "stated_peak_rph": 150, "requests_per_session": 5.0, "input_tokens": 4200, "output_tokens": 600, "cache_eligible": 0.28, "cache_realization": 0.35, "retry_rate": 0.07, "support_hours": 140, "budget": 18000, "headroom": 0.50, "policy_s": 8.0, "continuation_s": 18.0, "workflow_s": 12.0},
}

scenario_rows = []
for name, values in scenario_inputs.items():
    monthly_requests = values["sessions_per_day"] * values["requests_per_session"] * business_days_per_month
    implied_average_rph = values["sessions_per_day"] * values["requests_per_session"] / covered_hours_per_day
    conservative_rph = max(values["stated_peak_rph"], implied_average_rph)
    realized_hit_rate = values["cache_eligible"] * values["cache_realization"]
    billed_attempts = monthly_requests * (1 - realized_hit_rate) * (1 + values["retry_rate"])
    scenario_rows.append({**values, "scenario": name, "monthly_requests": monthly_requests, "implied_average_rph": implied_average_rph, "planning_rph": conservative_rph, "realized_hit_rate": realized_hit_rate, "billed_input_tokens": billed_attempts * values["input_tokens"], "billed_output_tokens": billed_attempts * values["output_tokens"]})

scenarios = pd.DataFrame(scenario_rows).set_index("scenario")
display(scenarios[["sessions_per_day", "monthly_requests", "stated_peak_rph", "implied_average_rph", "planning_rph", "realized_hit_rate", "retry_rate", "billed_input_tokens", "billed_output_tokens"]].round(2))
print("[Modeled] scenario outputs retain modeled status; no probability is assigned to low/expected/high.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), facecolor="#1a1a2e")
scenario_colors = ["#1e3a8a", "#1d4ed8", "#b45309"]
scenarios["monthly_requests"].plot.bar(ax=axes[0], color=scenario_colors)
axes[0].set_title("[Modeled] Monthly request range")
axes[0].set_ylabel("Requests")
(scenarios[["billed_input_tokens", "billed_output_tokens"]] / 1_000_000).plot.bar(ax=axes[1], color=["#1d4ed8", "#15803d"])
axes[1].set_title("[Modeled] Billed token range")
axes[1].set_ylabel("Million tokens")
for ax in axes:
    ax.set_facecolor("#1a1a2e")
    ax.tick_params(axis="x", rotation=0)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

scenario_checks = {
    "request_mix_sums_to_one": math.isclose(sum(request_mix.values()), 1.0),
    "sessions_are_ordered": scenarios.loc["low", "sessions_per_day"] < scenarios.loc["expected", "sessions_per_day"] < scenarios.loc["high", "sessions_per_day"],
    "tokens_are_positive": (scenarios[["input_tokens", "output_tokens"]] > 0).all().all(),
    "cache_rates_are_bounded": scenarios["realized_hit_rate"].between(0, 1).all(),
    "retry_rates_are_bounded": scenarios["retry_rate"].between(0, 1).all(),
    "planning_rate_covers_inputs": (scenarios["planning_rph"] >= scenarios[["stated_peak_rph", "implied_average_rph"]].max(axis=1)).all(),
}
for name, passed in scenario_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(scenario_checks.values()), "Stop: scenario model is internally inconsistent."

## 3 - Size Concurrency, Tokens, Quota, and Headroom

Little's Law supplies a floor: $L = \lambda W$. For planning, this notebook uses a request-mix-weighted service-time proxy, includes cache and retry effects, then applies explicit headroom. It does not claim that p95 service time inserted into Little's Law yields p95 concurrency. A burst/load test must validate the joint distribution.

**Predict:** will request quota or token quota be the tighter illustrative constraint in the high case? Resolve it from utilization, not intuition.

```mermaid
flowchart LR
    A["Planning arrivals"] --> D["Little's Law floor"]
    B["Request-mix service time"] --> D
    C["Cache and retries"] --> D
    D --> E["Add explicit headroom"]
    E --> F["Check RPM, TPM, concurrency"]
    F --> G["Admission and quota plan"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Check only RPM | Long prompts can exhaust TPM first |
| Wrong | Apply headroom twice because the frozen peak already mentions 50% headroom | Double counting destroys traceability |
| Wrong | Call Little's Law output a tail guarantee | It is a steady-state relationship, not a burst simulation |
| Right | Track RPM, TPM, concurrency, queue, and spend independently | Any dimension can become the admission limit |

**Quick Health Check:** capacity must cover the floor, quota dimensions must remain independent, and every placeholder quota must remain externally validated.

In [ ]:
# -- Model, plot, and check independent capacity dimensions ---------------
quota = {"rpm": 120, "tpm": 300_000, "concurrency": 20}  # [Modeled] Placeholder, not provider evidence.
capacity_rows = []
for name, row in scenarios.iterrows():
    uncached_service_s = request_mix["policy_lookup"] * row["policy_s"] + request_mix["editorial_continuation"] * row["continuation_s"] + request_mix["workflow_assistance"] * row["workflow_s"]
    effective_service_s = row["realized_hit_rate"] * 0.1 + (1 - row["realized_hit_rate"]) * uncached_service_s * (1 + row["retry_rate"])
    arrival_rps = row["planning_rph"] / 3600
    concurrency_floor = arrival_rps * effective_service_s
    planned_concurrency = max(1, math.ceil(concurrency_floor * (1 + row["headroom"])))
    peak_rpm = row["planning_rph"] / 60
    peak_tpm = peak_rpm * (1 - row["realized_hit_rate"]) * (1 + row["retry_rate"]) * (row["input_tokens"] + row["output_tokens"])
    capacity_rows.append({"scenario": name, "effective_service_s": effective_service_s, "concurrency_floor": concurrency_floor, "planned_concurrency": planned_concurrency, "peak_rpm": peak_rpm, "peak_tpm": peak_tpm, "rpm_utilization": peak_rpm / quota["rpm"], "tpm_utilization": peak_tpm / quota["tpm"], "concurrency_utilization": planned_concurrency / quota["concurrency"]})

capacity = pd.DataFrame(capacity_rows).set_index("scenario")
display(capacity.round(3))
high_driver = capacity.loc["high", ["rpm_utilization", "tpm_utilization", "concurrency_utilization"]].idxmax()
print(f"RESULT: high-case illustrative limiting dimension is {high_driver}.")
print("[External validation required] Replace all three quota placeholders with approved route/region evidence.")

utilization = capacity[["rpm_utilization", "tpm_utilization", "concurrency_utilization"]] * 100
ax = utilization.plot.bar(figsize=(10, 4.5), color=["#1d4ed8", "#b45309", "#15803d"])
ax.set_facecolor("#1a1a2e")
ax.figure.set_facecolor("#1a1a2e")
ax.axhline(100, color="#b91c1c", linestyle="--", label="Illustrative quota")
ax.set_title("[Modeled] Independent quota utilization")
ax.set_ylabel("Utilization (%)")
ax.tick_params(axis="x", rotation=0)
ax.spines[["top", "right"]].set_visible(False)
ax.legend()
plt.tight_layout()
plt.show()

capacity_checks = {
    "planned_concurrency_covers_floor": (capacity["planned_concurrency"] >= capacity["concurrency_floor"]).all(),
    "headroom_is_explicit": scenarios["headroom"].between(0, 1).all(),
    "token_rate_positive": (capacity["peak_tpm"] > 0).all(),
    "quota_dimensions_present": set(quota) == {"rpm", "tpm", "concurrency"},
}
for name, passed in capacity_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(capacity_checks.values())
print("LIMIT: queue depth, p99, cold starts, correlated retries, and burst duration need a load test or queueing simulation.")

## 4 - Price the Whole Service, Not Just Tokens

Token cost is visible and therefore easy to over-focus on. Riverside also pays for application infrastructure, retrieval and storage, observability and evaluation, other software, and support engineering. Failed attempts still bill; idle headroom can still cost money; security controls are never removed just to repair a spreadsheet.

```mermaid
flowchart TD
    A["Billed input/output tokens"] --> G["Monthly service envelope"]
    B["Application infrastructure"] --> G
    C["Retrieval and storage"] --> G
    D["Observability and evaluation"] --> G
    E["Other software"] --> G
    F["Support engineering"] --> G
    G --> H["Risk reserve"]
    H --> I["Budget and ceiling gates"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Quote token cost as service cost | Support and fixed operating cost can dominate |
| Wrong | Price only successful attempts | Failed and retried calls may still bill |
| Wrong | Hide contingency inside infrastructure | Reviewers cannot tell cost from uncertainty reserve |
| Right | Keep units, source date, currency, exclusions, and validation owner beside every rate | The estimate can be refreshed without reverse engineering |

**Quick Health Check:** every component appears exactly once, rates are externally validated placeholders, totals reconcile, and cost per request uses total service cost.

In [ ]:
# -- Attribute, plot, and reconcile monthly service cost ------------------
cost_inputs = pd.read_csv(COST_INPUT_PATH)
assert (cost_inputs["validation_status"] == "external_validation_required").all()

def rate_for(scenario, component):
    match = cost_inputs[(cost_inputs["scenario"] == scenario) & (cost_inputs["cost_component"] == component)]
    assert len(match) == 1, f"Expected one rate for {scenario}/{component}"
    return float(match.iloc[0]["rate"])

cost_rows = []
for name, row in scenarios.iterrows():
    model_input = row["billed_input_tokens"] / 1_000_000 * rate_for(name, "model_input")
    model_output = row["billed_output_tokens"] / 1_000_000 * rate_for(name, "model_output")
    components = {
        "model": model_input + model_output,
        "application_infrastructure": rate_for(name, "application_infrastructure"),
        "retrieval_and_storage": rate_for(name, "retrieval_and_storage"),
        "observability_and_evaluation": rate_for(name, "observability_and_evaluation"),
        "other_software": rate_for(name, "other_software"),
        "support_engineering": row["support_hours"] * rate_for(name, "support_engineering"),
    }
    subtotal = sum(components.values())
    components["risk_reserve"] = subtotal * 0.10
    components["total"] = subtotal + components["risk_reserve"]
    components["budget"] = row["budget"]
    components["cost_per_request"] = components["total"] / row["monthly_requests"]
    cost_rows.append({"scenario": name, **components})

costs = pd.DataFrame(cost_rows).set_index("scenario")
display(costs.round(2))
for name, row in costs.iterrows():
    status = "WITHIN" if row["total"] <= row["budget"] else "EXCEEDS"
    print(f"[Modeled] {name}: ${row['total']:,.0f}/month {status} scenario budget ${row['budget']:,.0f}; not a quote.")

component_columns = ["model", "application_infrastructure", "retrieval_and_storage", "observability_and_evaluation", "other_software", "support_engineering", "risk_reserve"]
ax = costs[component_columns].plot.bar(stacked=True, figsize=(11, 5), colormap="tab20")
ax.set_facecolor("#1a1a2e")
ax.figure.set_facecolor("#1a1a2e")
ax.scatter(range(len(costs)), costs["budget"], color="#b91c1c", marker="_", s=700, linewidths=3, label="Scenario budget")
ax.axhline(18_000, color="white", linestyle="--", label="Hard planning ceiling")
ax.set_title("[Modeled] Monthly service cost range - illustrative rates")
ax.set_ylabel("USD per month")
ax.tick_params(axis="x", rotation=0)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

recomputed_total = costs[component_columns].sum(axis=1)
cost_checks = {
    "totals_reconcile": np.allclose(recomputed_total, costs["total"]),
    "all_rates_have_sources": cost_inputs["source_or_basis"].notna().all(),
    "all_rates_have_dates": cost_inputs["source_date"].notna().all(),
    "all_rates_have_validation_owner": cost_inputs["validation_owner"].notna().all(),
}
for name, passed in cost_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(cost_checks.values())
print("LIMIT: taxes, discounts, FX, egress, service credits, incident surge, and customer-side cost remain excluded.")

## 5 - Sensitivity Before Optimization

The highest visible rate is not automatically the best optimization target. Change one assumption at a time around the expected case and measure the monthly effect. This is local sensitivity, not a probabilistic forecast; interactions and correlated high cases remain outside it.

```mermaid
flowchart LR
    A["Expected scenario"] --> B["Change one driver -25%"]
    A --> C["Change one driver +25%"]
    B --> D["Recompute total cost"]
    C --> D
    D --> E["Rank cost swing"]
    E --> F["Validate or optimize top driver"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Call a one-way sensitivity chart a confidence interval | No joint distribution or probability was modeled |
| Wrong | Optimize tokens before checking support or fixed cost | The largest controllable driver may be elsewhere |
| Wrong | Increase cache without an authorization key contract | Savings can create cross-scope leakage |
| Right | Pair each top driver with a validation experiment and a quality/security floor | Optimization remains evidence-led |

**Quick Health Check:** the baseline must reproduce expected cost, every perturbation changes one driver, and no sensitivity result is labeled probabilistic.

In [ ]:
# -- Compute, plot, and check one-way sensitivity -------------------------
expected = scenario_inputs["expected"].copy()
expected_rates = {component: rate_for("expected", component) for component in cost_inputs["cost_component"].unique()}

def expected_total(values, rates):
    requests = values["sessions_per_day"] * values["requests_per_session"] * business_days_per_month
    hit_rate = values["cache_eligible"] * values["cache_realization"]
    attempts = requests * (1 - hit_rate) * (1 + values["retry_rate"])
    model = attempts * (values["input_tokens"] * rates["model_input"] + values["output_tokens"] * rates["model_output"]) / 1_000_000
    subtotal = model + rates["application_infrastructure"] + rates["retrieval_and_storage"] + rates["observability_and_evaluation"] + rates["other_software"] + values["support_hours"] * rates["support_engineering"]
    return subtotal * 1.10

baseline_total = expected_total(expected, expected_rates)
drivers = ["support_hours", "requests_per_session", "input_tokens", "output_tokens", "retry_rate", "cache_realization", "model_input_rate", "support_hourly_rate", "application_infrastructure"]
sensitivity_rows = []
for driver in drivers:
    totals = {}
    for label, factor in [("minus_25", 0.75), ("plus_25", 1.25)]:
        values = expected.copy()
        rates = expected_rates.copy()
        if driver == "model_input_rate":
            rates["model_input"] *= factor
        elif driver == "support_hourly_rate":
            rates["support_engineering"] *= factor
        elif driver == "application_infrastructure":
            rates["application_infrastructure"] *= factor
        else:
            values[driver] *= factor
        totals[label] = expected_total(values, rates)
    sensitivity_rows.append({"driver": driver, "minus_25_delta": totals["minus_25"] - baseline_total, "plus_25_delta": totals["plus_25"] - baseline_total, "swing": totals["plus_25"] - totals["minus_25"]})

sensitivity = pd.DataFrame(sensitivity_rows).set_index("driver").sort_values("swing")
display(sensitivity.round(2))
print(f"[Modeled] baseline: ${baseline_total:,.0f}/month; largest local swing: {sensitivity['swing'].idxmax()}")

fig, ax = plt.subplots(figsize=(10, 5.5), facecolor="#1a1a2e")
ax.set_facecolor("#1a1a2e")
y = np.arange(len(sensitivity))
ax.barh(y, sensitivity["minus_25_delta"], color="#1d4ed8", label="Driver -25%")
ax.barh(y, sensitivity["plus_25_delta"], color="#b45309", label="Driver +25%")
ax.axvline(0, color="white", linewidth=1)
ax.set_yticks(y, sensitivity.index)
ax.set_xlabel("Change in modeled monthly cost (USD)")
ax.set_title("[Modeled] Local sensitivity around expected case")
ax.spines[["top", "right"]].set_visible(False)
ax.legend()
plt.tight_layout()
plt.show()

sensitivity_checks = {
    "baseline_matches_cost_table": math.isclose(baseline_total, costs.loc["expected", "total"], rel_tol=1e-9),
    "drivers_are_unique": sensitivity.index.is_unique,
    "both_directions_present": sensitivity[["minus_25_delta", "plus_25_delta"]].notna().all().all(),
}
for name, passed in sensitivity_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(sensitivity_checks.values())
print("ACTION: validate support hours/rate and fixed infrastructure before negotiating token discounts.")

## 6 - Map SLA Tiers to an Operating System

An SLA tier is not a row of percentages. It changes architecture, quota, monitoring, rollout, staffing, incident response, and price. Riverside's 99.5% target is scoped to covered service hours, while the sponsor also asks for deadline support whenever needed. That conflict must be resolved, not footnoted away.

```mermaid
flowchart LR
    A["Service definition"] --> B["SLIs and objectives"]
    B --> C["Architecture and quota"]
    C --> D["Monitoring and error budget"]
    D --> E["Rollout gates"]
    E --> F["Support and escalation"]
    F --> G["Commercial tier"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Promise 99.5% without covered hours and denominator | The same percentage can describe different obligations |
| Wrong | Trade security failures against ordinary availability error budget | Forbidden access is a separate zero-tolerance constraint |
| Wrong | Offer 24x7 response with weekday staffing | The support promise is operationally impossible |
| Right | Bind tier to architecture, quota, monitoring, rollout, support, and price | The commitment has an operating model |

**Quick Health Check:** every tier must specify all five operating dimensions, critical support must be funded, and policy constraints remain outside ordinary error-budget tradeoffs.

In [ ]:
# -- Derive, map, and check the covered-hours SLA tiers -------------------
availability_target = next(target["target"] for target in service_targets if target["name"] == "monthly_availability")
covered_hours_per_month = 10 * 5 * 52 / 12
allowed_unavailable_minutes = covered_hours_per_month * 60 * (1 - availability_target)

sla_tiers = pd.DataFrame([
    {"tier": "Pilot", "architecture": "One approved active route; disable switch; manual rollback", "quota": "Named cohort caps and hard spend/token limits", "monitoring": "Dashboard plus daily review; measure p50/p95/p99", "rollout": "Shadow then 12-editor canary", "support": "Best effort inside frozen covered hours", "offer_status": "Offerable as non-contractual pilot after security gates"},
    {"tier": "Business Hours", "architecture": "Redundant stateless app path; approved restore and fallback design", "quota": "Tenant RPM/TPM/concurrency/spend limits with validated headroom", "monitoring": "Burn, queue, throttle, retry, latency, and cost alerts", "rollout": "Canary and ramp gates with rollback owner", "support": "Severity response during 08:00-18:00 UK weekdays", "offer_status": "Provisional; load, quota, fallback, and terms validation required"},
    {"tier": "Critical", "architecture": "Fault-domain isolation and tested failover for approved state paths", "quota": "Reserved capacity and priority/load-shed policy", "monitoring": "24x7 paging, burn-rate response, tested incident evidence", "rollout": "Change windows, freeze rules, re-enablement authority", "support": "Staffed on-call and escalation explicitly priced", "offer_status": "Not offerable from current evidence or support agreement"},
]).set_index("tier")

print(f"[Modeled target] 99.5% across {covered_hours_per_month:.1f} covered hours allows about {allowed_unavailable_minutes:.1f} unavailable minutes/month.")
print("LIMIT: this arithmetic does not define exclusions, denominator, credits, or legal enforceability.")
display(sla_tiers)

tier_dimensions = ["architecture", "quota", "monitoring", "rollout", "support", "offer_status"]
sla_checks = {
    "all_tier_dimensions_present": sla_tiers[tier_dimensions].notna().all().all(),
    "availability_target_preserved": availability_target == fact_by_id["FACT-RIV-024"]["expected_value"],
    "covered_hours_preserved": case["support_and_handoff"]["covered_hours"] == fact_by_id["FACT-RIV-031"]["expected_value"],
    "critical_tier_not_offerable": "Not offerable" in sla_tiers.loc["Critical", "offer_status"],
    "security_not_spent_as_error_budget": next(target["target"] for target in service_targets if target["name"] == "authorization_leakage") == 0,
}
for name, passed in sla_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(sla_checks.values())

## 7 - Exercises: Break the Plan on Purpose

Each exercise changes one decision input and prints a check. A passing assertion means the calculation is internally consistent; it does not mean the commercial choice is approved.

```mermaid
flowchart LR
    A["Change one variable"] --> B["Recompute"]
    B --> C["Check budget or capacity gate"]
    C --> D["Name quality/security constraint"]
    D --> E["Record decision or validation work"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Your turn:** run each cell with the default, then change only the marked variable. Explain what evidence would justify adopting the cheaper setting.

In [ ]:
# -- Your turn: change one commercial driver at a time --------------------
CACHE_REALIZATION = 0.65  # CHANGE THIS: try 0.35 and 0.85.
OUTPUT_TOKEN_CAP = 220  # CHANGE THIS: compare 120, 220, and 600.
SUPPORT_HOURS = 80  # CHANGE THIS: compare 48, 80, and 140 hours/month.

cache_values = expected.copy()
cache_values["cache_realization"] = CACHE_REALIZATION
output_values = expected.copy()
output_values["output_tokens"] = OUTPUT_TOKEN_CAP
support_values = expected.copy()
support_values["support_hours"] = SUPPORT_HOURS

assert 0 <= CACHE_REALIZATION <= 1
assert 1 <= OUTPUT_TOKEN_CAP <= 600
assert SUPPORT_HOURS >= 0
print(f"[Modeled] cache realization {CACHE_REALIZATION:.0%}: ${expected_total(cache_values, expected_rates):,.0f}/month")
print(f"[Modeled] output cap {OUTPUT_TOKEN_CAP}: ${expected_total(output_values, expected_rates):,.0f}/month")
print(f"[Modeled] support hours {SUPPORT_HOURS}: ${expected_total(support_values, expected_rates):,.0f}/month")
print("GUARDRAIL: cache keys retain authorization scope; token caps rerun quality gates; support changes alter the service tier.")

## 8 - Commercial Decision and External Validation Register

The defensible output is a decision with conditions, not a decimal. Riverside can continue with a pilot planning envelope, but the current evidence does not support a critical tier or a final quote.

```mermaid
flowchart TD
    A["Modeled range"] --> D["Commercial decision record"]
    B["Measured synthetic mechanics"] --> D
    C["Policy constraints and unknowns"] --> D
    D --> E{"Quote prerequisites complete?"}
    E -->|No| F["Pilot or collect evidence"]
    E -->|Yes| G["Commercial approval"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Put the expected estimate in a proposal without range or expiry | It will be read as a promise |
| Wrong | Say 'subject to validation' without owners and evidence | The caveat has no completion path |
| Right | Name owner, evidence, decision, condition, and revalidation trigger | Unknowns become managed work |

**Quick Health Check:** the recommendation must fit current support and rollout evidence; every quote prerequisite has an owner; the high case crossing the ceiling triggers exception or scope change.

In [ ]:
# -- Preview and health-check the commercial decision --------------------
decision = {
    "status": "Approve pilot planning envelope; quote pending",
    "recommended_tier": "Pilot",
    "modeled_monthly_range_usd": [round(costs.loc["low", "total"]), round(costs.loc["high", "total"])],
    "expected_case_usd": round(costs.loc["expected", "total"]),
    "operating_target_usd": 14_000,
    "hard_planning_ceiling_usd": 18_000,
    "conditions": ["reconcile session and request demand", "validate request-type distributions under load", "validate price and billing units", "confirm quota and regional capacity", "test rollback/failover", "accept covered hours and severity response"],
}
validation_register = pd.DataFrame([
    ["Demand definitions and burst distribution", "Operations", "Shadow telemetry by request type and tenant tier"],
    ["Token and service-time distributions", "Platform Engineering", "Load test with long-context and retry slices"],
    ["Price, discount, tax, currency, billing unit", "Commercial Lead", "Dated approved price book or quote"],
    ["RPM, TPM, concurrency, regional capacity", "Platform Engineering", "Approved quota evidence per route and region"],
    ["Failover, restore, RTO/RPO", "SRE", "Failure injection and recovery evidence"],
    ["Support hours, severity response, escalation", "Support Lead", "Accepted staffing and service schedule"],
    ["SLA denominator, exclusions, credits, enforceability", "Contract Owner", "Reviewed contract language"],
], columns=["validation", "owner", "required_evidence"])

print(json.dumps(decision, indent=2))
display(validation_register)
print("DECISION: continue only as a bounded pilot estimate; do not issue a critical-tier commitment or quote.")

final_checks = {
    "three_scenarios_present": list(scenarios.index) == ["low", "expected", "high"],
    "average_only_conflict_visible": (scenarios["implied_average_rph"] > scenarios["stated_peak_rph"]).any(),
    "high_case_triggers_ceiling_action": costs.loc["high", "total"] > 18_000,
    "cost_includes_support": costs.loc["expected", "support_engineering"] > 0,
    "capacity_has_independent_limits": {"rpm_utilization", "tpm_utilization", "concurrency_utilization"}.issubset(capacity.columns),
    "critical_tier_withheld": decision["recommended_tier"] != "Critical",
    "external_validations_named": len(validation_register) >= 7,
}
for name, passed in final_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(final_checks.values()), "Stop: the planning package hides a material gap."
print("RESULT: the package is internally reviewable, still modeled, and explicitly not a quote.")

## 9 - Completed Roadmap, Coverage Ledger, and Key Takeaways

```mermaid
flowchart LR
    A["Average-only plan rejected"] --> B["Scenario range built"]
    B --> C["Capacity dimensions separated"]
    C --> D["Full cost attributed"]
    D --> E["Sensitivity ranked"]
    E --> F["Pilot tier recommended with conditions"]
    style A fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Completed roadmap

- The 62-session/hour average was rejected because it hides request/session conversion and burst shape; the temporary planning rate is the larger conflicting input, not a production measurement.
- Low, expected, and high scenarios preserve modeled status and expose token, cache, retry, and demand sensitivity.
- RPM, TPM, concurrency, queue, and spend remain independent admission dimensions; illustrative quota placeholders stay externally unvalidated.
- Full service cost includes infrastructure, retrieval/storage, observability/evaluation, support, failed attempts, and idle headroom.
- The high case crosses the 18,000 USD planning ceiling, so the commercial response is exception review, scope change, or more evidence rather than a smaller decimal.
- Riverside can discuss a bounded pilot planning envelope; it cannot issue a critical-tier commitment or quote from this notebook.

### Reflection bridges across the failure chain

| What the step fixed | Residual failure that forced the next step |
|---|---|
| Reconciled sessions/hour with requests/hour | One reconciled point still hid plausible demand and token ranges |
| Built low/expected/high stories | Volume alone did not identify the limiting quota dimension |
| Separated RPM, TPM, and concurrency | Capacity still omitted the operated service's full cost |
| Attributed the whole service | A total did not reveal which assumption controlled the decision |
| Ranked one-way sensitivity | A cheaper configuration could still weaken quality, security, support, or rollout |
| Mapped service tiers | A modeled tier still was not a quote or customer commitment |

### Coverage ledger

| Tier | Techniques |
|---|---|
| Built and measured on synthetic/local evidence | Trace attribution mechanics, fixture integrity checks |
| Built and modeled | Arrival reconciliation, scenario volume, token/cache/retry amplification, Little's Law floor, headroom, RPM/TPM/concurrency checks, full cost attribution, sensitivity, error-budget arithmetic, SLA tier map |
| Named with external validation required | Live prices, discounts, taxes, quota, regional capacity, load tails, cache safety, failover, RTO/RPO, staffing, service credits, legal language |

If you find a technique named above that does not appear in the tier table, that is exactly the bug this section exists to catch.

### Key takeaways

1. An average is a conservation check, not a burst-capacity plan.
2. Little's Law gives a concurrency floor; tails, bursts, and correlated retries still need load evidence.
3. Cache eligibility is not cache realization, and neither overrides authorization scope.
4. Check RPM, TPM, concurrency, queue, and spend independently.
5. Price the operated service, including support and observability, not just successful model tokens.
6. A scenario range is not a probability distribution.
7. An SLA tier is architecture, quota, monitoring, rollout, and support joined to measurable terms.
8. A modeled estimate becomes a quote only after named commercial and technical validation.

> **Forward:** carry the selected pilot tier, cohort caps, rollback conditions, support hours, and unresolved validation register into `06-rollout-and-change-management`.